# Yanai-band momentum and energy flux — TPOSE6

Repeat of the TPOSE24 analysis on the TPOSE6 assimilation (TPOSE-Vel), daily
output 2012-09-01 .. 2013-06-30 (303 days). Differences from TPOSE24:

- **Whole record used** — the run has no spin-up, so none is dropped. The
  20-day band-pass edge-trim is still applied (filter transient, not spin-up).
- **Daily** sampling (band-pass 15–40 day, daily).
- **PHIHYD reconstructed** from THETA/SALT via JMD95Z (no PHIHYD diagnostic in
  this run); `rhonil = 1035`.
- **Profiles at 170°W, 140°W and 110°W** (all now inside the domain); the
  profile figures loop over longitude.
- **Maps** cover 175°W–105°W, 10°S–10°N.

In [1]:
import sys, os
sys.path.append('/home/edavenport/analysis/motive-yanai-waves/scripts')
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
import wave_filter as wf, fluxes as fx, tpose6_io as io

fx.RHO0 = io.RHONIL                      # energy flux uses rhonil = 1035
CACHE   = io.CACHE_DIR
FIG_DIR = '/home/edavenport/analysis/motive-yanai-waves/figures/tpose6'
os.makedirs(FIG_DIR, exist_ok=True)

prof  = xr.open_dataset(f'{CACHE}/yanai_profiles_tp6.nc')
cov3d = xr.open_dataset(f'{CACHE}/yanai_flux_cov3d_tp6.nc')
maps  = xr.open_dataset(f'{CACHE}/yanai_flux_maps_tp6.nc')

# No spin-up drop for TPOSE6: the whole record is the filter input.
LONS = [str(x) for x in prof['lon'].values]      # 170W,140W,110W
LATS = [str(x) for x in prof['lat'].values]      # 1S,0N,1N,2N,3N
LON_DEG = prof['lon_deg'].values
LAT_DEG = prof['lat_deg'].values
Z    = prof['depth'].values                      # cell-center depth (m, <0)
drF  = prof['drF'].values
hFac = prof['hFacC'].values                      # (lon, lat, depth)

# After filtering, keep only the interior: drop the first/last EDGE_TRIM_DAYS of
# the filtered signal (band-pass edge contamination near each record boundary).
time_all = prof['time'].values
etm      = io.edge_trim_mask(time_all)           # interior-keeping mask
time     = time_all[etm]                         # analysis time window
print('profile lons =', dict(zip(LONS, np.round(LON_DEG, 2))),
      'lats =', dict(zip(LATS, np.round(LAT_DEG, 2))))
print(f'filter input: {time_all.size} daily steps (no spin-up drop); '
      f'analysis window after {io.EDGE_TRIM_DAYS:.0f}-day edge trim: '
      f'{str(time[0])[:10]} to {str(time[-1])[:10]} ({time.size} steps)')

profile lons = {'170W': np.float64(189.92), '140W': np.float64(219.92), '110W': np.float64(249.92)} lats = {'1S': np.float64(-0.92), '0N': np.float64(-0.08), '1N': np.float64(0.92), '2N': np.float64(2.08), '3N': np.float64(2.92)}
filter input: 303 daily steps (no spin-up drop); analysis window after 20-day edge trim: 2012-09-21 to 2013-06-10 (263 steps)


## Band-pass filtering and flux components at the profile columns

Columns have dims (time, lon, lat, depth). WVEL is already averaged to cell
centers and PHIHYD is already reconstructed in the cache, so here we only
band-pass (15–40 day, daily) and form the flux products.

In [2]:
# 15-40 day daily band-pass of each field along time (axis 0), then keep the
# edge-trimmed interior (etm); columns -> (time, lon, lat, depth)
def bp(name):
    return wf.bandpass(prof[name].values, axis=0, dt_days=io.DT_DAYS)[etm]

up   = bp('UVEL')
vp   = bp('VVEL')
wp   = bp('WVEL')          # already on cell centers in the cache
phip = bp('PHIHYD')

# instantaneous flux components (time, lon, lat, depth)
flux = fx.momentum_fluxes(up, vp, wp)           # uu,vv,ww,uv,uw,vw  (m^2 s^-2)
flux.update(fx.energy_fluxes(phip, up, vp, wp)) # pu,pv,pw           (W m^-2)

# time-mean profiles and standard deviation over the window (lon, lat, depth)
fmean = {k: np.nanmean(v, axis=0) for k, v in flux.items()}
fstd  = {k: np.nanstd(v,  axis=0) for k, v in flux.items()}

# vertical convergence of the vertical momentum fluxes (lon, lat, depth), m s^-2
conv = {k: fx.vertical_convergence(fmean[k], Z, zaxis=-1) for k in ('uw', 'vw')}

# background (unfiltered) time-mean velocity over the same window (lon,lat,depth)
Ubg = np.nanmean(prof['UVEL'].values[etm], axis=0)
Vbg = np.nanmean(prof['VVEL'].values[etm], axis=0)

# full-column, partial-cell-weighted depth integral (time,lon,lat,depth)->(time,lon,lat)
def depth_integral(fld):
    thick = drF[None, None, None, :] * hFac[None, :, :, :]
    return np.sum(np.where(np.isfinite(fld), fld, 0.0)
                  * np.where(np.isfinite(fld), thick, 0.0), axis=-1)

/tmp/claude-1040720/ipykernel_2465853/1837976614.py:16: RuntimeWarning: Mean of empty slice
  fmean = {k: np.nanmean(v, axis=0) for k, v in flux.items()}
/home/edavenport/miniforge3/envs/tpose/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:2015: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/claude-1040720/ipykernel_2465853/1837976614.py:23: RuntimeWarning: Mean of empty slice
  Ubg = np.nanmean(prof['UVEL'].values[etm], axis=0)
/tmp/claude-1040720/ipykernel_2465853/1837976614.py:24: RuntimeWarning: Mean of empty slice
  Vbg = np.nanmean(prof['VVEL'].values[etm], axis=0)


## (a) Time series — full-column depth-integrated flux, per longitude

In [3]:
ts = {k: depth_integral(flux[k]) for k in ('pw', 'uw', 'vw', 'uv')}  # (time,lon,lat)
# shared symmetric y-limit per row across all longitudes (for comparison)
ROW_YLIM = {k: float(np.nanmax(np.abs(ts[k]))) for k in ('pw', 'uw', 'vw', 'uv')}
labels = {'pw': r"$\int \rho_0\,\phi' w'\,dz$  (W m$^{-1}$)",
          'uw': r"$\int u'w'\,dz$  (m$^3$ s$^{-2}$)",
          'vw': r"$\int v'w'\,dz$  (m$^3$ s$^{-2}$)",
          'uv': r"$\int u'v'\,dz$  (m$^3$ s$^{-2}$)"}

def timeseries_fig(li, LON):
    fig, axes = plt.subplots(4, 1, figsize=(11, 11), sharex=True)
    for ax, k in zip(axes, ('pw', 'uw', 'vw', 'uv')):
        for c, L in enumerate(LATS):
            ax.plot(time, ts[k][:, li, c], lw=0.9, label=L)
        ax.axhline(0, color='k', lw=0.5)
        ax.set_ylabel(labels[k])
        ax.set_ylim(-ROW_YLIM[k], ROW_YLIM[k])
    axes[0].legend(ncol=5, fontsize=8, loc='upper right', title='latitude')
    loc = mdates.AutoDateLocator(maxticks=8)
    axes[-1].xaxis.set_major_locator(loc)
    axes[-1].xaxis.set_major_formatter(mdates.ConciseDateFormatter(loc))
    axes[-1].set_xlabel('date')
    axes[0].set_title(f'Full-column depth-integrated 15-40 day flux at {LON}')
    fig.tight_layout()
    fig.savefig(f'{FIG_DIR}/a1_flux_timeseries_{LON}.png', dpi=140)
    plt.close(fig)

for li, LON in enumerate(LONS):
    timeseries_fig(li, LON)
print('saved a1 time series for', LONS)

saved a1 time series for ['170W', '140W', '110W']


## (a2) Depth–time (Hovmöller) sections of the vertical fluxes, per longitude

In [4]:
comps_ts = [('uw', r"$u'w'$", 'm$^2$ s$^{-2}$'),
            ('vw', r"$v'w'$", 'm$^2$ s$^{-2}$'),
            ('pw', r"$\rho_0\phi' w'$", 'W m$^{-2}$')]
# shared color scale per component across all longitudes (for comparison)
HOV_V98 = {k: float(np.nanpercentile(np.abs(flux[k]), 98)) for k, _, _ in comps_ts}
# colorbar-limit tweaks: shrink u'w' & v'w' by 25%, grow rho0*phi'w' by 25%
HOV_SCALE = {'uw': 0.75, 'vw': 0.75, 'pw': 1.25}
HOV_T = mdates.date2num(time)          # date numbers for contourf x-axis

def _seafloor(li, c):
    # bottom-face depth (m, <0) of the deepest wet cell at this (lon, lat)
    wet = np.where(hFac[li, c] > 0)[0]
    kb = int(wet[-1]) if wet.size else len(Z) - 1
    return Z[kb] - drF[kb] / 2.0

def hovmoller_fig(li, LON):
    # columns share the y-axis (tick labels only on the left, so nothing
    # overlaps); the shared axis runs to the deepest seafloor among the shown
    # latitudes. Each column's own bottom depth is noted in its title.
    bot = [_seafloor(li, c) for c in range(len(LATS))]
    ybot = min(bot)                            # deepest (most negative)
    fig, axes = plt.subplots(len(comps_ts), len(LATS), figsize=(16, 9),
                             sharex=True, sharey=True)
    for r, (k, sym, unit) in enumerate(comps_ts):
        v98 = HOV_V98[k] * HOV_SCALE[k]        # shared across lon, tweaked per row
        lev = np.linspace(-v98, v98, 101)      # >=100 filled-contour levels
        for c, L in enumerate(LATS):
            ax = axes[r, c]
            m = ax.contourf(HOV_T, Z, flux[k][:, li, c, :].T, levels=lev,
                            cmap='RdBu_r', extend='both')
            if r == 0: ax.set_title(f'{L} ({-bot[c]:.0f} m)')
            if c == 0: ax.set_ylabel(sym + '\ndepth (m)', fontsize=9)
        cb = fig.colorbar(m, ax=axes[r, :].tolist(), shrink=0.8, pad=0.01)
        cb.set_label(unit, fontsize=9)
        cb.set_ticks(np.linspace(-v98, v98, 5))    # avoid 100-tick clutter
        cb.formatter.set_powerlimits((-2, 2)); cb.update_ticks()
    axes[0, 0].set_ylim(ybot, 0)               # sharey propagates to all panels
    for ax in axes[-1]:
        loc = mdates.AutoDateLocator(maxticks=6)
        ax.xaxis.set_major_locator(loc)
        ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(loc))
        ax.tick_params(axis='x', labelsize=8); ax.set_xlabel('date')
    fig.suptitle(f'15-40 day vertical fluxes vs depth and time at {LON}')
    fig.savefig(f'{FIG_DIR}/a2_flux_hovmoller_{LON}.png', dpi=140, bbox_inches='tight')
    plt.close(fig)

for li, LON in enumerate(LONS):
    hovmoller_fig(li, LON)
print('saved a2 Hovmoller for', LONS)

saved a2 Hovmoller for ['170W', '140W', '110W']


## (b) Mean profiles with standard deviation, per longitude

In [5]:
def _linthresh(*arrs, ratio=1e-3):
    v = np.concatenate([np.abs(a[np.isfinite(a)]).ravel() for a in arrs])
    mx = v.max() if v.size else 1.0
    return max(mx * ratio, 1e-15)

# Shared x-scale per (row, latitude) across ALL longitudes, so the same panel
# is directly comparable between the 170W/140W/110W figures.
_ROW_SRC = {
    'vel':  [(Ubg, None), (Vbg, None)],
    'uwvw': [(fmean['uw'], fstd['uw']), (fmean['vw'], fstd['vw'])],
    'uv':   [(fmean['uv'], fstd['uv'])],
    'conv': [(conv['uw'], None), (conv['vw'], None)],
    'pw':   [(fmean['pw'], fstd['pw'])],
}
def _lat_ext(items, c):
    e = []
    for m, s in items:
        d = np.abs(m[:, c, :]) + (0.0 if s is None else np.nan_to_num(s[:, c, :]))
        if np.isfinite(d).any(): e.append(np.nanmax(d))
    return max(e) if e else 1.0

def _tidy_xaxis(ax, symlog=False, lt=None):
    if symlog:
        ax.set_xscale('symlog', linthresh=lt, linscale=0.5)
        ax.xaxis.set_major_locator(mticker.SymmetricalLogLocator(
            linthresh=lt, base=100, subs=(1.0,)))
    else:
        ax.xaxis.set_major_locator(mticker.MaxNLocator(nbins=4))
        ax.ticklabel_format(axis='x', style='sci', scilimits=(-2, 2))
        ax.xaxis.get_offset_text().set_fontsize(7)
    ax.tick_params(axis='x', labelsize=7, rotation=30)

def _set_xscale(ax, rowkey, c, symlog, xcrop):
    # apply the shared (across-longitude) x scaling for this row & latitude
    items = _ROW_SRC[rowkey]
    if symlog:
        _tidy_xaxis(ax, True, _linthresh(*[m[:, c, :] for m, s in items]))
    else:
        _tidy_xaxis(ax)
        ext = _lat_ext(items, c)
        frac = xcrop if (xcrop and rowkey != 'vel') else 1.05
        ax.set_xlim(-frac * ext, frac * ext)

def profile_panels(li, LON, symlog=False, zoom=None, xcrop=None, fname=None, suffix=''):
    fig, axes = plt.subplots(5, len(LATS), figsize=(16, 15), sharey=True)
    for c, L in enumerate(LATS):
        ax = axes[0, c]
        ax.plot(Ubg[li, c], Z, label='u'); ax.plot(Vbg[li, c], Z, label='v')
        ax.axvline(0, color='k', lw=0.5); ax.set_title(L)
        _set_xscale(ax, 'vel', c, False, xcrop)
        if c == 0: ax.set_ylabel('time-mean velocity (m s$^{-1}$)\ndepth (m)')
        if c == len(LATS)-1: ax.legend(fontsize=8)
        ax = axes[1, c]
        for k, col in (('uw', 'C0'), ('vw', 'C1')):
            ax.plot(fmean[k][li, c], Z, col, label=k[0]+"'"+k[1]+"'")
            ax.fill_betweenx(Z, fmean[k][li, c]-fstd[k][li, c],
                             fmean[k][li, c]+fstd[k][li, c], color=col, alpha=0.15)
        ax.axvline(0, color='k', lw=0.5)
        _set_xscale(ax, 'uwvw', c, symlog, xcrop)
        if c == 0: ax.set_ylabel(r"$\langle u'w'\rangle,\langle v'w'\rangle\pm1\sigma$"
                                 "\n(m$^2$ s$^{-2}$)\ndepth (m)")
        if c == len(LATS)-1: ax.legend(fontsize=8)
        ax = axes[2, c]
        ax.plot(fmean['uv'][li, c], Z, 'C2', label="u'v'")
        ax.fill_betweenx(Z, fmean['uv'][li, c]-fstd['uv'][li, c],
                         fmean['uv'][li, c]+fstd['uv'][li, c], color='C2', alpha=0.15)
        ax.axvline(0, color='k', lw=0.5)
        _set_xscale(ax, 'uv', c, symlog, xcrop)
        if c == 0: ax.set_ylabel(r"$\langle u'v'\rangle\pm1\sigma$"
                                 "\n(m$^2$ s$^{-2}$)\ndepth (m)")
        if c == len(LATS)-1: ax.legend(fontsize=8)
        ax = axes[3, c]
        ax.plot(conv['uw'][li, c], Z, 'C0', label=r"$-\partial_z\langle u'w'\rangle$")
        ax.plot(conv['vw'][li, c], Z, 'C1', label=r"$-\partial_z\langle v'w'\rangle$")
        ax.axvline(0, color='k', lw=0.5)
        _set_xscale(ax, 'conv', c, symlog, xcrop)
        if c == 0: ax.set_ylabel('mom.-flux convergence\n(m s$^{-2}$)\ndepth (m)')
        if c == len(LATS)-1: ax.legend(fontsize=8)
        ax = axes[4, c]
        ax.plot(fmean['pw'][li, c], Z, 'C3')
        ax.fill_betweenx(Z, fmean['pw'][li, c]-fstd['pw'][li, c],
                         fmean['pw'][li, c]+fstd['pw'][li, c], color='C3', alpha=0.15)
        ax.axvline(0, color='k', lw=0.5)
        _set_xscale(ax, 'pw', c, symlog, xcrop)
        if c == 0: ax.set_ylabel(r"$\langle\rho_0\phi' w'\rangle\pm1\sigma$"
                                 "\n(W m$^{-2}$)\ndepth (m)")
    axes[0, 0].set_ylim((-zoom[1], -zoom[0]) if zoom else (-3000, 0))
    fig.suptitle(f'15-40 day flux profiles and background velocity at {LON}' + suffix, y=0.997)
    fig.tight_layout()
    if fname: fig.savefig(f'{FIG_DIR}/{fname}', dpi=140)
    plt.close(fig)

for li, LON in enumerate(LONS):
    profile_panels(li, LON, xcrop=0.30, fname=f'b_profiles_{LON}.png',
                   suffix=' (full depth, linear flux, x cropped to 30%)')
    profile_panels(li, LON, zoom=(0, 1500), xcrop=0.30,
                   fname=f'b_profiles_zoom_0-1500m_{LON}.png',
                   suffix=' (0-1500 m zoom, linear flux, x cropped to 30%)')
    profile_panels(li, LON, symlog=True, fname=f'b_profiles_symlogflux_{LON}.png',
                   suffix=' (symlog flux axis)')
print('saved b profiles for', LONS)

saved b profiles for ['170W', '140W', '110W']


### (b overlay) 170°W / 140°W / 110°W overlaid at 0N, 1N, 2N

Same flux quantities as above, but each panel overlays the three longitudes
(coloured) so the along-equator change is directly visible. x-limits are shared
per row across the three latitudes shown.

In [6]:
def profile_overlay(lat_sel=('0N', '1N', '2N'), xcrop=0.5, zoom=None,
                    fname='b_profiles_overlay_0N1N2N.png', suffix=''):
    rows = [
        ('u',  'background u\n(m s$^{-1}$)',                  Ubg,         None,        False),
        ('uw', r"$\langle u'w'\rangle$"'\n(m$^2$ s$^{-2}$)', fmean['uw'], fstd['uw'],  True),
        ('vw', r"$\langle v'w'\rangle$"'\n(m$^2$ s$^{-2}$)', fmean['vw'], fstd['vw'],  True),
        ('uv', r"$\langle u'v'\rangle$"'\n(m$^2$ s$^{-2}$)', fmean['uv'], fstd['uv'],  True),
        ('pw', r"$\langle\rho_0\phi' w'\rangle$"'\n(W m$^{-2}$)', fmean['pw'], fstd['pw'], True),
    ]
    cidx = [LATS.index(L) for L in lat_sel]
    lon_colors = ['#1b9e77', '#d95f02', '#7570b3']    # 170W, 140W, 110W
    fig, axes = plt.subplots(len(rows), len(lat_sel),
                             figsize=(4.2*len(lat_sel), 15), sharey=True)
    for r, (key, lab, mean, std, crop) in enumerate(rows):
        # shared symmetric x-limit for this row across shown lats & all lons
        ext = np.nanmax([np.abs(mean[:, c, :]) + (0.0 if std is None else
                         np.nan_to_num(std[:, c, :])) for c in cidx])
        lim = (xcrop if crop else 1.05) * ext
        for cc, (c, L) in enumerate(zip(cidx, lat_sel)):
            ax = axes[r, cc]
            for li, LON in enumerate(LONS):
                ax.plot(mean[li, c], Z, color=lon_colors[li], lw=1.3, label=LON)
                if std is not None:
                    ax.fill_betweenx(Z, mean[li, c]-std[li, c], mean[li, c]+std[li, c],
                                     color=lon_colors[li], alpha=0.10)
            ax.axvline(0, color='k', lw=0.5); ax.set_xlim(-lim, lim); _tidy_xaxis(ax)
            if r == 0: ax.set_title(L)
            if cc == 0: ax.set_ylabel(lab + '\ndepth (m)')
            if r == 0 and cc == len(lat_sel)-1: ax.legend(fontsize=8, title='longitude')
    axes[0, 0].set_ylim((-zoom[1], -zoom[0]) if zoom else (-3000, 0))
    fig.suptitle('15-40 day flux profiles - 170W/140W/110W overlaid at '
                 + ', '.join(lat_sel) + suffix, y=0.999)
    fig.tight_layout()
    fig.savefig(f'{FIG_DIR}/{fname}', dpi=140)
    plt.close(fig)

profile_overlay(suffix=' (full depth, x cropped to 50%)')
profile_overlay(zoom=(0, 1500), fname='b_profiles_overlay_0N1N2N_zoom_0-1500m.png',
                suffix=' (0-1500 m zoom, x cropped to 50%)')
print('saved b overlay figures')

saved b overlay figures


### Energy-flux component profiles, per longitude

In [7]:
def energy_flux_profiles(li, LON):
    rows = [('pu', 'C0', r"$\langle\rho_0\phi' u'\rangle\pm1\sigma$"),
            ('pv', 'C1', r"$\langle\rho_0\phi' v'\rangle\pm1\sigma$"),
            ('pw', 'C3', r"$\langle\rho_0\phi' w'\rangle\pm1\sigma$")]
    fig, axes = plt.subplots(3, len(LATS), figsize=(16, 10), sharey=True)
    for r, (k, col, lab) in enumerate(rows):
        for c, L in enumerate(LATS):
            ax = axes[r, c]
            ax.plot(fmean[k][li, c], Z, col)
            ax.fill_betweenx(Z, fmean[k][li, c]-fstd[k][li, c],
                             fmean[k][li, c]+fstd[k][li, c], color=col, alpha=0.15)
            ax.axvline(0, color='k', lw=0.5); _tidy_xaxis(ax)
            # shared symmetric x-limit across longitudes for this (component, lat)
            ext = np.nanmax(np.abs(fmean[k][:, c, :]) + np.nan_to_num(fstd[k][:, c, :]))
            ax.set_xlim(-ext, ext)
            if r == 0: ax.set_title(L)
            if c == 0: ax.set_ylabel(lab + '\n(W m$^{-2}$)\ndepth (m)')
    axes[0, 0].set_ylim(-3000, 0)
    fig.suptitle(f'15-40 day energy-flux component profiles at {LON}', y=0.995)
    fig.tight_layout()
    fig.savefig(f'{FIG_DIR}/b_energy_flux_profiles_{LON}.png', dpi=140)
    plt.close(fig)

for li, LON in enumerate(LONS):
    energy_flux_profiles(li, LON)
print('saved energy-flux profiles for', LONS)

saved energy-flux profiles for ['170W', '140W', '110W']


## (c) Maps — depth-averaged, time-mean flux (175°W–105°W, 10°S–10°N)

Partial-cell-weighted **depth average** within each layer (native flux units),
so the unequal-thickness layers are comparable on a shared color scale.

Black crosses mark the profile columns (170°W, 140°W, 110°W × 1S,0N,1N,2N,3N).

In [8]:
lon = maps['lon'].values; lat = maps['lat'].values
LAYERS = [str(x) for x in maps['layer'].values]
# profile-column markers: every (profile lon, profile lat) pair
mk_lon = np.repeat(LON_DEG, len(LAT_DEG))
mk_lat = np.tile(LAT_DEG, len(LON_DEG))

def map_grid(components, comp_labels, unit, fname, title, cmap='RdBu_r', diverging=True):
    nr, nc = len(components), len(LAYERS)
    fig, axes = plt.subplots(nr, nc, figsize=(3.0*nc, 2.6*nr + 0.6),
                             sharex=True, sharey=True, squeeze=False,
                             layout='constrained')
    for r, comp in enumerate(components):
        row = maps['flux_avg'].sel(component=comp).values      # (layer, y, x)
        v98 = np.nanpercentile(np.abs(row), 98)
        for c, lay in enumerate(LAYERS):
            ax = axes[r, c]
            if diverging:
                m = ax.pcolormesh(lon, lat, row[c], cmap=cmap,
                                  vmin=-v98, vmax=v98, shading='auto')
            else:
                m = ax.pcolormesh(lon, lat, row[c], cmap=cmap,
                                  vmin=0, vmax=v98, shading='auto')
            ax.axhline(0, color='k', lw=0.4)
            ax.plot(mk_lon, mk_lat, 'kx', ms=3)
            if r == 0: ax.set_title(lay, fontsize=9)
            if c == 0: ax.set_ylabel(comp_labels[r] + '\nlat', fontsize=9)
        cb = fig.colorbar(m, ax=axes[r, :].tolist(), shrink=0.85, pad=0.01)
        cb.set_label(unit, fontsize=9)
    for ax in axes[-1]: ax.set_xlabel('lon (°E)')
    fig.suptitle(title)
    fig.savefig(f'{FIG_DIR}/{fname}', dpi=140)
    plt.close(fig)

In [9]:
# Energy flux components: rho0 * phi' * (u', v', w')
map_grid(['pu', 'pv', 'pw'],
         [r"$\overline{\rho_0\phi' u'}$", r"$\overline{\rho_0\phi' v'}$",
          r"$\overline{\rho_0\phi' w'}$"],
         'W m$^{-2}$', 'c1_energy_flux_maps.png',
         'Depth-averaged time-mean 15-40 day energy flux (W m$^{-2}$)')

In [10]:
# Vertical momentum-flux components: u'w', v'w'
map_grid(['uw', 'vw'],
         [r"$\overline{u'w'}$", r"$\overline{v'w'}$"],
         'm$^2$ s$^{-2}$', 'c2_vertical_momflux_maps.png',
         'Depth-averaged time-mean 15-40 day vertical momentum flux (m$^2$ s$^{-2}$)')

In [11]:
# u'v' (signed) and the velocity variances u'u', v'v', w'w'
map_grid(['uv'], [r"$\overline{u'v'}$"], 'm$^2$ s$^{-2}$',
         'c3_uv_momflux_maps.png',
         'Depth-averaged time-mean 15-40 day u\'v\' momentum flux (m$^2$ s$^{-2}$)')
map_grid(['uu', 'vv', 'ww'],
         [r"$\overline{u'u'}$", r"$\overline{v'v'}$", r"$\overline{w'w'}$"],
         'm$^2$ s$^{-2}$', 'c4_variance_maps.png',
         'Depth-averaged time-mean 15-40 day velocity variances (m$^2$ s$^{-2}$)',
         cmap='viridis', diverging=False)
print('saved maps c1-c4')

saved maps c1-c4
